In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1784534223976_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


In [2]:
df = spark.read.parquet('s3://airline-dataset-2020-2025/Silver/')

In [6]:
rows = df.count()
cols = len(df.columns)
print(rows, cols)

(40910253, 120)

In [7]:
df.printSchema()

root
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- DOT_ID_Marketing_Airline: integer (nullable = true)
 |-- IATA_Code_Marketing_Airline: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- DOT_ID_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- IATA_Code_Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- Flight_Num_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- DOT_ID_Operating_Airline: integer (nullable = true)
 |-- IATA_Code_Operating_Airline: string (nullable = true)


In [8]:
df = df.select('FlightDate','Year','Month','Quarter','DayOfMonth','DayOfWeek','Marketing_Airline_Network','Origin','OriginState','Dest','DestState','CRSDepTime','ArrDelay','ArrDel15','DepDelay','DepDel15','CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay','AirTime','Cancelled','Diverted','Distance','TaxiOut','TaxiIn')

### 1. Unique Identifiers

Examples

Flight_Number_Marketing_Airline
Tail_Number
OriginAirportID
OriginAirportSeqID
DestAirportID
DOT_ID_Operating_Airline

These simply identify flights or airports.

Example

Tail Number = N934AA

This doesn't help discover patterns unless you're specifically studying aircraft utilization.

### 2. Duplicate Information

Example

Origin = ATL
OriginCityName = Atlanta, GA
OriginState = Georgia
OriginAirportID = 10397
OriginAirportSeqID = 1039705
OriginCityMarketID = 30397
OriginStateFips = 13
OriginWac = 34

For most analyses,

Origin
OriginState

are sufficient.

The IDs are useful only for joining with other datasets.

### 3. Code Share Columns

Examples

Originally_Scheduled_Code_Share_Airline
IATA_Code_Originally_Scheduled_Code_Share_Airline
DOT_ID_Originally_Scheduled_Code_Share_Airline

These are relevant mainly to airline operations or alliance studies, not delay analysis.

### 4. Actual Time Columns

Examples

DepTime
ArrTime
WheelsOff
WheelsOn
CRSArrTime

These can be useful, but for a first-level analysis:

DepDelay already tells how late departure was.
ArrDelay already tells arrival delay.

Unless you're calculating turnaround time or hourly traffic, these timestamps are often unnecessary.

### 5. Redundant Delay Columns

Example

DepDelay
DepDelayMinutes
DepartureDelayGroups

Suppose

DepDelay = 24

DepDelayMinutes = 24

DepartureDelayGroups = 2

They all represent the same information in different formats.

Keeping

DepDelay
DepDel15

is usually enough.

### 6. Diversion Columns

Examples

Div1Airport
Div2Airport
Div3Airport
Div4Airport
Div5Airport
Div1WheelsOn
Div1TailNum
...

These are mostly empty because very few flights are diverted.

For example,

99.8% NULL

0.2% have values

Keeping them increases memory usage without adding much value to general EDA.

### 7. Cancellation Code
CancellationCode

Useful only if

Cancelled == 1

Since most flights are not cancelled, this column is mostly null.

### 8. Flights Column
Flights = 1

Almost every record has a value of 1.

It provides no additional information.

### 9. Duplicate Column
Duplicate

Usually a data-quality indicator added during preprocessing.

Rarely useful for analysis.

### 10. Extra Column
_c119

Looks like an accidental import column.

Should be dropped.

In [9]:
from pyspark.sql.functions import col, isnan, when, count

missing = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

missing.show(vertical=True)

-RECORD 0-----------------------------
 FlightDate                | 0        
 Year                      | 0        
 Month                     | 0        
 Quarter                   | 0        
 DayOfMonth                | 0        
 DayOfWeek                 | 0        
 Marketing_Airline_Network | 0        
 Origin                    | 0        
 OriginState               | 0        
 Dest                      | 0        
 DestState                 | 0        
 CRSDepTime                | 0        
 ArrDelay                  | 1014879  
 ArrDel15                  | 1014879  
 DepDelay                  | 896518   
 DepDel15                  | 896518   
 CarrierDelay              | 33265986 
 WeatherDelay              | 33265986 
 NASDelay                  | 33265986 
 SecurityDelay             | 33265986 
 LateAircraftDelay         | 33265986 
 AirTime                   | 1014879  
 Cancelled                 | 0        
 Diverted                  | 0        
 Distance                

In [10]:
numeric_cols = [
'DepDelay',
'ArrDelay',
'CarrierDelay',
'WeatherDelay',
'NASDelay',
'SecurityDelay',
'LateAircraftDelay',
'Distance',
'AirTime',
'TaxiOut',
'TaxiIn'
]

df.select(numeric_cols).describe().show()

+-------+------------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+-----------------+------------------+-----------------+-----------------+
|summary|          DepDelay|         ArrDelay|      CarrierDelay|     WeatherDelay|          NASDelay|      SecurityDelay| LateAircraftDelay|         Distance|           AirTime|          TaxiOut|           TaxiIn|
+-------+------------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+-----------------+------------------+-----------------+-----------------+
|  count|          40013735|         39895374|           7644267|          7644267|           7644267|            7644267|           7644267|         40910253|          39895374|         39997567|         39983837|
|   mean|10.937542346396805|5.242046383623324|25.263957028188575|4.279215521906809|13.147254144838216|0.13603436405347955| 27.12500452953828

In [11]:
df.groupBy("Marketing_Airline_Network") \
.count() \
.orderBy(desc("count")) \
.show()

+-------------------------+--------+
|Marketing_Airline_Network|   count|
+-------------------------+--------+
|                       AA|10407897|
|                       DL| 8526129|
|                       WN| 7582834|
|                       UA| 7427635|
|                       AS| 2238407|
|                       B6| 1366470|
|                       NK| 1278352|
|                       F9|  968030|
|                       G4|  694895|
|                       HA|  419604|
+-------------------------+--------+

In [12]:
df.groupBy("Origin") \
.count() \
.orderBy(desc("count")) \
.show(20)

+------+-------+
|Origin|  count|
+------+-------+
|   ATL|1915120|
|   ORD|1778804|
|   DFW|1705528|
|   DEN|1683092|
|   CLT|1347556|
|   LAX|1086362|
|   SEA|1026212|
|   PHX|1021572|
|   LAS| 999915|
|   IAH| 904228|
|   MCO| 859437|
|   LGA| 824195|
|   DTW| 771724|
|   EWR| 754423|
|   SFO| 744221|
|   BOS| 734638|
|   DCA| 734587|
|   MSP| 722032|
|   SLC| 661333|
|   JFK| 657831|
+------+-------+
only showing top 20 rows

In [13]:
df.groupBy("Dest") \
.count() \
.orderBy(desc("count")) \
.show(20)

+----+-------+
|Dest|  count|
+----+-------+
| ATL|1914971|
| ORD|1778590|
| DFW|1705415|
| DEN|1682968|
| CLT|1347494|
| LAX|1086362|
| SEA|1026130|
| PHX|1021494|
| LAS| 999980|
| IAH| 904123|
| MCO| 859424|
| LGA| 824201|
| DTW| 771676|
| EWR| 754431|
| SFO| 744383|
| DCA| 734667|
| BOS| 734643|
| MSP| 722020|
| SLC| 661231|
| JFK| 657671|
+----+-------+
only showing top 20 rows

In [14]:
df.groupBy("OriginState") \
.count() \
.orderBy(desc("count")) \
.show()

+-----------+-------+
|OriginState|  count|
+-----------+-------+
|         TX|4409615|
|         CA|4089241|
|         FL|3456538|
|         IL|2320934|
|         GA|2078926|
|         NY|1988710|
|         NC|1913405|
|         CO|1898418|
|         VA|1543198|
|         WA|1235694|
|         AZ|1185323|
|         NV|1123267|
|         PA|1026228|
|         MI|1000306|
|         TN| 834680|
|         NJ| 782096|
|         MN| 765516|
|         MA| 758879|
|         MO| 708015|
|         UT| 699784|
+-----------+-------+
only showing top 20 rows

In [15]:
df.groupBy("DestState") \
.count() \
.orderBy(desc("count")) \
.show()

+---------+-------+
|DestState|  count|
+---------+-------+
|       TX|4409467|
|       CA|4089330|
|       FL|3456663|
|       IL|2320696|
|       GA|2078780|
|       NY|1988609|
|       NC|1913388|
|       CO|1898292|
|       VA|1543279|
|       WA|1235629|
|       AZ|1185274|
|       NV|1123332|
|       PA|1026139|
|       MI|1000308|
|       TN| 834736|
|       NJ| 782122|
|       MN| 765503|
|       MA| 758893|
|       MO| 708043|
|       UT| 699676|
+---------+-------+
only showing top 20 rows

In [16]:
df.groupBy("Year") \
.count() \
.orderBy("Year") \
.show()

+----+-------+
|Year|  count|
+----+-------+
|2020|5022397|
|2021|6311871|
|2022|7013508|
|2023|7278739|
|2024|7546968|
|2025|7736770|
+----+-------+

In [17]:
df.groupBy("Quarter") \
.count() \
.orderBy("Quarter") \
.show()

+-------+--------+
|Quarter|   count|
+-------+--------+
|      1|10169608|
|      2| 9847864|
|      3|10549499|
|      4|10343282|
+-------+--------+

In [18]:
df.groupBy("Month") \
.count() \
.orderBy("Month") \
.show()

+-----+-------+
|Month|  count|
+-----+-------+
|    1|3358992|
|    2|3141722|
|    3|3668894|
|    4|3246144|
|    5|3249065|
|    6|3352655|
|    7|3617203|
|    8|3590730|
|    9|3341566|
|   10|3525386|
|   11|3378386|
|   12|3439510|
+-----+-------+

In [19]:
df.select(avg("DepDelay")).show()

+------------------+
|     avg(DepDelay)|
+------------------+
|10.937542346396805|
+------------------+

In [20]:
df.select(avg("ArrDelay")).show()

+-----------------+
|    avg(ArrDelay)|
+-----------------+
|5.242046383623324|
+-----------------+

In [21]:
df.select(
max("DepDelay"),
max("ArrDelay")
).show()

+-------------+-------------+
|max(DepDelay)|max(ArrDelay)|
+-------------+-------------+
|       7223.0|       7232.0|
+-------------+-------------+

In [22]:
df.select(
min("DepDelay"),
min("ArrDelay")
).show()

+-------------+-------------+
|min(DepDelay)|min(ArrDelay)|
+-------------+-------------+
|       -131.0|       -139.0|
+-------------+-------------+

In [23]:
df.groupBy("Marketing_Airline_Network") \
.agg(
avg("DepDelay").alias("Avg_Dep_Delay"),
avg("ArrDelay").alias("Avg_Arr_Delay")
) \
.orderBy(desc("Avg_Dep_Delay")) \
.show()

+-------------------------+------------------+------------------+
|Marketing_Airline_Network|     Avg_Dep_Delay|     Avg_Arr_Delay|
+-------------------------+------------------+------------------+
|                       B6|18.101130534949778|12.147421932830499|
|                       F9|17.310434781689196| 12.81508360285754|
|                       G4|14.446877516294759|12.900632680867137|
|                       NK|13.779831078326515| 8.252641895449296|
|                       AA| 12.04873140455054| 7.142431625209059|
|                       WN|11.347636617314812| 4.195521054100276|
|                       UA|10.991397635390664|5.2767419763279815|
|                       DL| 8.322598569262272|1.5880958090213946|
|                       HA| 6.306052359361556| 5.036483844113604|
|                       AS| 5.334017514554565|2.3903048525880286|
+-------------------------+------------------+------------------+

In [24]:
df.groupBy("OriginState") \
.agg(avg("DepDelay").alias("AvgDelay")) \
.orderBy(desc("AvgDelay")) \
.show()

+-----------+------------------+
|OriginState|          AvgDelay|
+-----------+------------------+
|         DE|22.955696202531644|
|         PR|15.653040404401075|
|         WY|15.426579458060807|
|         WV|15.183275471698114|
|         NJ|14.456749898968484|
|         CO|14.011275656602212|
|         MD|13.923538296814653|
|         FL|13.860263776369413|
|         TX|13.068227690754902|
|         IL|12.665866689396974|
|         ND|12.635875546708267|
|         NY|12.120776222542723|
|         NV| 12.03275124049469|
|         SD| 11.92480660258184|
|         ME|11.699694795186135|
|         VI|11.631999786774701|
|         MA|11.628990396041743|
|         LA|11.499390414264306|
|         VT|11.426215132787707|
|         VA|11.130566503872842|
+-----------+------------------+
only showing top 20 rows

In [25]:
df.groupBy("Origin") \
.agg(avg("DepDelay").alias("AvgDelay")) \
.orderBy(desc("AvgDelay")) \
.show(20)

+------+------------------+
|Origin|          AvgDelay|
+------+------------------+
|   VRB|44.292682926829265|
|   MGW|40.493860845839016|
|   PPG|31.903614457831324|
|   HGR|27.243203526818515|
|   HYA| 25.35228331780056|
|   ASE| 25.25720889280211|
|   SCK| 24.53170447934846|
|   BIH|23.701310043668123|
|   MMH|23.597989949748744|
|   SMX|23.255502392344496|
|   USA| 23.12608353033885|
|   ILG|22.955696202531644|
|   CKB|22.355499527261266|
|   LAF|21.948051948051948|
|   LCK| 20.79664363277394|
|   OTH|20.505218216318784|
|   OGS|  20.1595818815331|
|   COD| 19.66096866096866|
|   BQN| 19.57188498402556|
|   MVY|19.322878228782287|
+------+------------------+
only showing top 20 rows

In [26]:
df.select(
avg("CarrierDelay"),
avg("WeatherDelay"),
avg("NASDelay"),
avg("SecurityDelay"),
avg("LateAircraftDelay")
).show()

+------------------+-----------------+------------------+-------------------+----------------------+
| avg(CarrierDelay)|avg(WeatherDelay)|     avg(NASDelay)| avg(SecurityDelay)|avg(LateAircraftDelay)|
+------------------+-----------------+------------------+-------------------+----------------------+
|25.263957028188575|4.279215521906809|13.147254144838216|0.13603436405347955|     27.12500452953828|
+------------------+-----------------+------------------+-------------------+----------------------+

In [27]:
df.groupBy("DepDel15").count().show()

+--------+--------+
|DepDel15|   count|
+--------+--------+
|     0.0|32448294|
|    null|  896518|
|     1.0| 7565441|
+--------+--------+

In [28]:
df.groupBy("Cancelled") \
.count() \
.withColumn(
"Percentage",
round(col("count")*100/df.count(),2)
).show()

+---------+--------+----------+
|Cancelled|   count|Percentage|
+---------+--------+----------+
|      0.0|39993169|     97.76|
|      1.0|  917084|      2.24|
+---------+--------+----------+

In [30]:
numeric = [
'DepDelay',
'ArrDelay',
'CarrierDelay',
'WeatherDelay',
'NASDelay',
'LateAircraftDelay',
'Distance',
'TaxiOut',
'TaxiIn',
'AirTime'
]

for col_name in numeric:
    print("Correlation with ArrDelay:", col_name)
    print(df.stat.corr(col_name,"ArrDelay"))

('Correlation with ArrDelay:', 'DepDelay')
0.962665738554
('Correlation with ArrDelay:', 'ArrDelay')
1.0
('Correlation with ArrDelay:', 'CarrierDelay')
0.684713587076
('Correlation with ArrDelay:', 'WeatherDelay')
0.299708111079
('Correlation with ArrDelay:', 'NASDelay')
0.34177250678
('Correlation with ArrDelay:', 'LateAircraftDelay')
0.607217135418
('Correlation with ArrDelay:', 'Distance')
-0.00132847705906
('Correlation with ArrDelay:', 'TaxiOut')
0.188455516851
('Correlation with ArrDelay:', 'TaxiIn')
0.109493495122
('Correlation with ArrDelay:', 'AirTime')
0.0163457159928

In [31]:
df.groupBy("Origin","Dest") \
.agg(avg("ArrDelay").alias("AvgDelay")) \
.orderBy(desc("AvgDelay")) \
.show(20)

+------+----+------------------+
|Origin|Dest|          AvgDelay|
+------+----+------------------+
|   BUR| FLL|             922.0|
|   CVG| CLE|             817.0|
|   JFK| LGA|             755.0|
|   PSP| CLT|             716.0|
|   GJT| ATL| 433.6666666666667|
|   JAX| TPA|             348.0|
|   FLL| PIE|             321.0|
|   AVL| USA|             310.0|
|   CLL| MIA|             262.0|
|   ACT| BHM|             255.0|
|   CLL| GNV|             223.0|
|   BIL| IFP|             222.0|
|   AUS| SAF|            206.25|
|   IAH| BUF|             204.0|
|   CLT| ASE|             185.0|
|   RSW| DSM|             171.5|
|   OKC| CAE|             171.0|
|   LAX| ATW|             164.0|
|   PIE| VPS|159.33333333333334|
|   EUG| OKC|             159.0|
+------+----+------------------+
only showing top 20 rows

In [32]:
df.groupBy("Origin","Dest") \
.agg(avg("ArrDelay").alias("AvgDelay")) \
.orderBy(desc("AvgDelay")) \
.show(20)

+------+----+------------------+
|Origin|Dest|          AvgDelay|
+------+----+------------------+
|   BUR| FLL|             922.0|
|   CVG| CLE|             817.0|
|   JFK| LGA|             755.0|
|   PSP| CLT|             716.0|
|   GJT| ATL| 433.6666666666667|
|   JAX| TPA|             348.0|
|   FLL| PIE|             321.0|
|   AVL| USA|             310.0|
|   CLL| MIA|             262.0|
|   ACT| BHM|             255.0|
|   CLL| GNV|             223.0|
|   BIL| IFP|             222.0|
|   AUS| SAF|            206.25|
|   IAH| BUF|             204.0|
|   CLT| ASE|             185.0|
|   RSW| DSM|             171.5|
|   OKC| CAE|             171.0|
|   LAX| ATW|             164.0|
|   PIE| VPS|159.33333333333334|
|   EUG| OKC|             159.0|
+------+----+------------------+
only showing top 20 rows

In [33]:
df = df.withColumn(
"DepHour",
floor(col("CRSDepTime")/100)
)

In [34]:
df.groupBy("DepHour") \
.count() \
.orderBy("DepHour") \
.show()

+-------+-------+
|DepHour|  count|
+-------+-------+
|      0|  65646|
|      1|  22789|
|      2|   8367|
|      3|   5516|
|      4|   2844|
|      5| 966534|
|      6|2784505|
|      7|2818033|
|      8|2822422|
|      9|2400548|
|     10|2613552|
|     11|2613967|
|     12|2500545|
|     13|2483819|
|     14|2463873|
|     15|2409964|
|     16|2370098|
|     17|2597656|
|     18|2489378|
|     19|2165741|
+-------+-------+
only showing top 20 rows

In [29]:
print("Total Flights:", df.count())

print("Cancelled Flights:",
df.filter(col("Cancelled")==1).count())

print("Diverted Flights:",
df.filter(col("Diverted")==1).count())

print("Delayed Flights (>15 mins):",
df.filter(col("DepDel15")==1).count())

print("Average Arrival Delay:",
df.select(avg("ArrDelay")).first()[0])

print("Average Departure Delay:",
df.select(avg("DepDelay")).first()[0])

('Total Flights:', 40910253)
('Cancelled Flights:', 917084)
('Diverted Flights:', 97790)
('Delayed Flights (>15 mins):', 7565441)
('Average Arrival Delay:', 5.242046383623324)
('Average Departure Delay:', 10.937542346396805)